<a href="https://colab.research.google.com/github/Kavyaavula16/IT-TICKETING-SYSTEM/blob/main/Copy_of_Customer_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
thoughtvector_customer_support_on_twitter_path = kagglehub.dataset_download('thoughtvector/customer-support-on-twitter')

print('Data source import complete.')


Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.
Data source import complete.


## Customer Sentiment Analysis
Feng Li, fengl2, 2019/10/08

I’m working for an IT company in technical support team. We’re dealing with different issues
from various customer in a daily basis. We’re using ticketing system and live chat to
communicate with customers. Customer feedback is very import to us – When we see positive
feedbacks, we know our service is good and we may be able to expand our business with them.
When we see negative feedbacks, we need to take action to understand the reason and
address customer’s concerns timely.

So, we need to carefully monitor customer’s sentiment in all communications between our
support engineers and customers. Basically, we want to do at least two things 1) track
customer’s satisfaction level over times and give action suggestions; 2) real time monitor
ongoing communications and raise alarms when necessary.

However, our company’s data cannot be shared in public. So, this project will be using similar
data - “Customer Support on Twitter” dataset from Kaggle. And according to the effort and
limited time of this course, this project will focus on the first task “track customer’s satisfaction
level over times and give action suggestions”.

This notebook includes code and document.

This notebook is being developed as Kaggle kernel using dataset https://www.kaggle.com/thoughtvector/customer-support-on-twitter and usig this kernel https://www.kaggle.com/soaxelbrooke/customer-sentiment-by-brand as reference.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/customer-support-on-twitter/sample.csv
/kaggle/input/customer-support-on-twitter/twcs/twcs.csv


In [ ]:
# load libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from tqdm import tqdm_notebook
from datetime import datetime

In [ ]:
# Load customer feedback data from kaggle dataset
tweets = pd.read_csv('sample.csv')
tweets.shape

(93, 7)

In [ ]:
tweets.columns

Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

Columns Meaning
- tweet_id

The unique ID for this tweet
- author_id

The unqiue ID for this tweet author (anonymized for non-company users)
- inbound

Whether or not the tweet was sent (inbound) to a company
- created_at

When the tweet was created
- text

The text content of the tweet

In [ ]:
# We'll focus on what customers said to us, so let's get only customer messages
customer_msg = tweets[tweets.inbound]
customer_msg.shape

(49, 7)

In [ ]:
# Original dataset includes support data from multiple companies. Here we choose one company with similar business as ours.
# for example Sprint.
customer_msg = customer_msg[customer_msg['text'].str.contains("sprintcare")]
customer_msg.shape

(0, 7)

In [ ]:
# Let's take a look at the data
customer_msg.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id


In [ ]:
# Reduce data size for better performance purpose in testing
customer_msg_sample = customer_msg.head(1000)
customer_msg_sample.shape

(0, 7)

In [ ]:
tqdm_notebook().pandas()

/tmp/ipykernel_961/1334260503.py:1: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  tqdm_notebook().pandas()


0it [00:00, ?it/s]

In [ ]:
# Change data type to be datetime for column "created_at" and
# sort by the ascending order so we can analize the messages based on the time when they are created
customer_msg_sample['created_at'] = pd.to_datetime(customer_msg_sample.created_at)
customer_msg_sample = customer_msg_sample.sort_values(by='created_at')

In [ ]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
# Instantiate sentiment analyzer from NLTK, make "sentiment_analyze" function
sentiment_analyzer = SentimentIntensityAnalyzer()

def sentiment_analyze(text: str) -> float:
    return sentiment_analyzer.polarity_scores(text)['compound']


In [ ]:
# Analyze customer sentiment based on their messages
customer_msg_sample['sentiment'] = \
    customer_msg_sample.text.progress_apply(sentiment_analyze)

0it [00:00, ?it/s]

In [ ]:
customer_msg_sample.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,sentiment


In [ ]:
# Group by author_id so we can deal with messages from different customers(author_id)
customer_grouped = customer_msg_sample.groupby('author_id')

In [ ]:
# Case 1, we can calculate average sentiment values for each customer over this data chunk
# and compare to predefined alert threshold.
# If the average sentiment value is lower than the threshold we'll list all the customers(author_id)
# and suggest to reach to them for further discussion to find the reason and resolve the issues proactively
author_sentiment_avg = customer_grouped.sentiment.mean().sort_values()
author_sentiment_avg_df = pd.DataFrame({'author_id':author_sentiment_avg.index, 'sentiment':author_sentiment_avg.values})
alert_threshold_avg = -0.7
author_sentiment_avg_df[author_sentiment_avg_df.sentiment <= alert_threshold_avg]

,author_id,sentiment


In [ ]:
# Case 2, we can find the minimum sentiment values for each customer over this data chunk
# and compare to predefined alert threshold.
# If the minimum sentiment value is lower than the threshold we'll list all the customers(author_id)
# and suggest to reach to them for further discussion to find the reason and resolve the issues proactively
author_sentiment_lowest = customer_grouped.sentiment.min().sort_values()
author_sentiment_lowest_df = pd.DataFrame({'author_id':author_sentiment_lowest.index, 'sentiment':author_sentiment_lowest.values})
alert_threshold_min = -0.8
author_sentiment_lowest_df[author_sentiment_lowest_df.sentiment <= alert_threshold_min]

,author_id,sentiment
